## Classical ML Model #3: **Gradient Boosting Classifier**

First, we will import our pre-split and scaled training, validation, and test sets.

In [9]:
import pandas as pd
import joblib

# Load the preprocessed and scaled data
X_train_scaled = joblib.load('processed_data/X_train_scaled.pkl')
X_val_scaled = joblib.load('processed_data/X_val_scaled.pkl')
X_test_scaled = joblib.load('processed_data/X_test_scaled.pkl')

y_train = joblib.load('processed_data/y_train.pkl')
y_val = joblib.load('processed_data/y_val.pkl')
y_test = joblib.load('processed_data/y_test.pkl')

# **[4]** Model Selection Training


Using Scipy-kit as library

Link to documentations
- https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.GradientBoostingClassifier.html#sklearn.ensemble.GradientBoostingClassifier.get_params
- https://xgboost.readthedocs.io/en/release_3.2.0/python/python_intro.html

In [4]:
import xgboost as xgb
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.model_selection import RandomizedSearchCV
import scipy.stats as stats

print(f'XGBoost Version: {xgb.__version__}')

XGBoost Version: 3.2.0


In [5]:
def assess_metrics(y_true, y_pred, mode):
    accuracy = accuracy_score(y_true, y_pred)
    print(f'Accuracy in {mode} set = {round(accuracy * 100, 4)} %')
    print()
    print(f'predicted: {list(y_pred)}')
    print(f'   actual: {list(y_true)}')

    if (mode == 'test'):
        print(f'Classification report')
        print(f'{classification_report(y_true, y_pred)}')

### 3.1 Training Phase

In [6]:
param_dist = {
    'max_depth': 3,
    'learning_rate': 0.01,
    'subsample': 0.5,
    'n_estimators': 20
}

xgb_model = xgb.XGBClassifier(**param_dist)
xgb_model.fit(X_train_scaled, y_train)
y_pred = xgb_model.predict(X_train_scaled)

assess_metrics(y_train, y_pred, 'train')

Accuracy in train set = 61.7305 %

predicted: [np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.i

# **[5]** Error Analysis and Model Tuning

Credits
- https://medium.com/@dicee/optimizing-xgboost-a-guide-to-hyperparameter-tuning-77b6e48e289d 
- https://www.geeksforgeeks.org/machine-learning/xgbclassifier/

In [7]:
# Defining hyperparameter 
param_dist = {
    'max_depth': stats.randint(3, 10),
    'learning_rate': stats.uniform(0.01, 0.1),
    'subsample': stats.uniform(0.5, 0.5),
    'n_estimators': stats.randint(20, 100)
}

# Defining randoms and training it
random_search = RandomizedSearchCV(xgb_model, param_distributions=param_dist, n_iter=10, cv=5, scoring='accuracy', random_state=42)
random_search.fit(X_val_scaled, y_val)

# Displaying best set of hyperparameters and corresponding score
best_params = random_search.best_params_
print(f'Best set of hyperparameters: {random_search.best_params_}')
print(f'Best score: {round(random_search.best_score_ * 100, 4)} %')

xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(X_val_scaled, y_val)
y_pred = xgb_model.predict(X_val_scaled)
assess_metrics(y_val, y_pred, 'val')

Best set of hyperparameters: {'learning_rate': np.float64(0.06986584841970366), 'max_depth': 9, 'n_estimators': 94, 'subsample': np.float64(0.7296244459829335)}
Best score: 80.916 %
Accuracy in val set = 97.1457 %

predicted: [np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int

# **[6]** Model Evaluation

In [8]:
xgb_model = xgb.XGBClassifier(**best_params)
xgb_model.fit(X_test_scaled, y_test)
y_pred = xgb_model.predict(X_test_scaled)

assess_metrics(y_test, y_pred, 'test')

Accuracy in test set = 97.0959 %

predicted: [np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.in

<hr/>

# AI Declaration
Chavez, Allen Visagar
- sample text

Llanes, Andre Gabriel De Ocampo
- sample text

Rojo, Von Matthew De Guzman *(leader)*
- Uses Google-search that has AI features in synthesizing different sources relevant to questions I asked.
- Uses Google Gemini to ask ways on how to import CSV dataset and scaffolding ideas for EDA

Tan, Jeremy James Teves
- sample text